In [ ]:
test_ids = ['P11473', 'Q91XI3', 'hello', 'ENSG00000157764', 'ENSG00000139618']
result = main(test_ids)
result

In [ ]:
import requests
import json
import re
import pandas as pd
import numpy as np

UNIPROT_BASE_URL = "https://rest.uniprot.org/uniprotkb"
ENSEMBL_BASE_URL = "https://rest.ensembl.org"


def http_function(endpoint, method="GET", headers=None, params=None):
    if method == "GET":
        return requests.get(endpoint, headers=headers, params=params)


def get_uniprot(accession):
    endpoint = f"{UNIPROT_BASE_URL}/{accession}.json"
    http_args = {"headers": {"Accept": "application/json"}}
    return http_function(endpoint, **http_args)


def uniprot_parse_response(resp):
    status = resp.status_code

    if status != 200:
        data = resp.json()
        msg_list = data.get("messages")
        if isinstance(msg_list, list) and msg_list:
            message = msg_list[0]
        else:
            message = data.get("error") or json.dumps(data)
        return {"error": message}

    data = resp.json()
    accession = data.get("primaryAccession") or data.get("uniProtkbId")
    organism = None
    org_block = data.get("organism") or {}
    if isinstance(org_block, dict):
        organism = org_block.get("scientificName")
    genes = data.get("genes") or []

    seq_block = data.get("sequence") or {}
    sequence_info = {
        "value": seq_block.get("value"),
        "length": seq_block.get("length"),
        "molWeight": seq_block.get("molWeight"),
        "crc64": seq_block.get("crc64"),
        "md5": seq_block.get("md5"),
    }

    entry_type = data.get("entryType") or "protein"

    return {
        accession: {
            "organism": organism,
            "geneInfo": genes,
            "sequenceInfo": sequence_info,
            "type": entry_type,
        }
    }

def get_ensembl(id_):
    endpoint = f"{ENSEMBL_BASE_URL}/lookup/id/{id_}"
    http_args = {"headers": {"Accept": "application/json"}, "params": {"expand": 0}}
    return http_function(endpoint, **http_args)


def ensembl_parse_response(resp):
    status = resp.status_code
    
    if status != 200:
    
        data = resp.json()
        message = data.get("error") or json.dumps(data)
       
        return {"error": message}

    data = resp.json()

    ensembl_id = data.get("id")

    fields = [
        "object_type",
        "species",
        "assembly_name",
        "biotype",
        "display_name",
        "id",
        "db_type",
        "description",
        "canonical_transcript",
        "source",
    ]
    parsed = {key: data.get(key) for key in fields}
    return {ensembl_id: parsed}

UNIPROT_RE = re.compile(r"^(?:[OPQ][0-9][A-Z0-9]{3}[0-9]|[A-NR-Z][0-9]{5})$")
ENSEMBL_RE = re.compile(r"^ENS[A-Z0-9]+[0-9]+$")

def main(ids):
    results = {}

    for _id in ids:
        if UNIPROT_RE.match(_id):
            resp = get_uniprot(_id)
            parsed = uniprot_parse_response(resp)
            if "error" in parsed and len(parsed) == 1:
                results[_id] = parsed["error"]
            else:
                results.update(parsed)
        elif ENSEMBL_RE.match(_id):
            resp = get_ensembl(_id)
            parsed = ensembl_parse_response(resp)
            if "error" in parsed and len(parsed) == 1:
                results[_id] = parsed["error"]
            else:
                results.update(parsed)
        else:
            results[_id] = "error:unknown database"

    return results
